# What a certified symmetrizing norm buys a linear solver

The manuscript states that the certified operators admit a symmetric positive-definite
formulation and the reference does not, and supports it with one number: a weighted-symmetry
residual of $1.2\times10^{-16}$ against $0.52$. This notebook turns that sentence into an
experiment.

The question is not which solver is fastest. It is which solver is *available*, and what
happens if you use one that is not. Three routes are attempted for every operator:

1. **`cg_symmetric`** — conjugate gradients on $-Q_IL_I$. Legitimate only when that matrix is
   genuinely symmetric positive definite, which is what a certified weighted extended-Gauss
   identity delivers.
2. **`cg_forced`** — conjugate gradients on the symmetric *part* of $-Q_IL_I$. This always
   runs and always returns something. When the symmetry residual is not negligible it has
   quietly solved a different problem, and the resulting solution error measures the cost of
   pretending.
3. **`gmres` / `bicgstab`** — non-symmetric iterations on $L_I$ itself, the honest practical
   route when no symmetric system exists.

No model calls, no search. Everything is recomputed from the released operator arrays; runtime
is well under a minute.

> **Caveat carried through the whole notebook.** Iteration counts are not comparable *across*
> solver families: BiCGSTAB uses two matrix-vector products per iteration and restarted GMRES
> rebuilds its Krylov basis. The comparison that matters here is availability and correctness,
> not a speed race.


## 1. Load the release package

Identical loading to the downstream/novelty notebook: the exact candidates from
`derived/operators`, the follow-up operators, and the MOLE/Corbino–Castillo references from the
run archives, de-duplicated by program identity. This is the only cell that defines
`operators`.

In [2]:
import inspect, json, math, re, shutil, zipfile
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.sparse.linalg import bicgstab, cg, gmres

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

OUTPUT_DIR = Path("solver_experiment")
for folder in ("tables", "figures"): (OUTPUT_DIR / folder).mkdir(parents=True, exist_ok=True)


@dataclass
class Operator:
    name: str
    family: str            # 'reference' | 'candidate' | 'promoted' | 'followup'
    cells: int
    D: np.ndarray
    G: np.ndarray
    Q: np.ndarray | None
    P: np.ndarray | None
    metadata: dict = field(default_factory=dict)

    @property
    def L(self) -> np.ndarray:
        return self.D @ self.G

    def interior_laplacian(self) -> np.ndarray:
        """Dirichlet interior block: drop the two boundary scalar unknowns."""
        return self.L[1:-1, 1:-1]

    def interior_norm(self) -> np.ndarray | None:
        if self.Q is None: return None
        return self.Q[1:-1, 1:-1]


def _as_matrix(value) -> np.ndarray | None:
    if value is None: return None
    arr = np.asarray(value, dtype=float)
    return np.diag(arr) if arr.ndim == 1 else arr


def load_operators(root: Path) -> list[Operator]:
    """Every operator archive under one directory. Provenance is assigned by the caller."""
    operators: list[Operator] = []
    for path in sorted(Path(root).rglob("*_m*.npz")):
        with np.load(path, allow_pickle=True) as bundle:
            keys = set(bundle.files)
            if not {"D", "G"} <= keys: continue
            metadata = {}
            if "metadata_json" in keys:
                try: metadata = json.loads(str(bundle["metadata_json"]))
                except Exception: metadata = {}
            operators.append(Operator(
                name=path.stem, family="candidate", cells=int(bundle["cells"]),
                D=np.asarray(bundle["D"], dtype=float), G=np.asarray(bundle["G"], dtype=float),
                Q=_as_matrix(bundle["Q"] if "Q" in keys else (bundle["q"] if "q" in keys else None)),
                P=_as_matrix(bundle["P"] if "P" in keys else (bundle["p"] if "p" in keys else None)),
                metadata=metadata))
    return operators


PACKAGE_ROOT = Path("Verifier_Guided_Mimetic_Operators_Reproducibility_Package_Final_Clean")
if not PACKAGE_ROOT.exists():
    bundles = sorted(Path(".").glob("*Reproducibility_Package_Final_Clean*.zip"))
    if not bundles:
        raise FileNotFoundError(
            "Put the release zip beside this notebook, or set PACKAGE_ROOT to the unpacked folder."
        )
    with zipfile.ZipFile(bundles[-1]) as bundle: bundle.extractall(".")
    PACKAGE_ROOT = next(p for p in Path(".").glob("*Reproducibility_Package_Final_Clean*")
                        if p.is_dir())

# The MOLE references live only inside the run archives; unpack them once.
RUN_CACHE = PACKAGE_ROOT / "runs" / "_unpacked"
for archive in sorted((PACKAGE_ROOT / "runs").glob("*.zip")):
    target = RUN_CACHE / archive.stem
    if not target.exists():
        with zipfile.ZipFile(archive) as bundle: bundle.extractall(target)

SOURCE_ROOTS = [
    (PACKAGE_ROOT / "derived" / "operators", "promoted"),
    (PACKAGE_ROOT / "derived" / "followup_operators", "followup"),
    (RUN_CACHE / "v11_original_results", None),   # references and closed-grid candidates
]


def classify(stem: str, default: str | None) -> str:
    if "reference_mole" in stem: return "reference"
    return default or "candidate"


def program_key(operator: Operator) -> str:
    """Identify the same operator across naming conventions.

    The release stores a paper candidate as `<condition>_prog_...` while the run archive
    stores the same arrays as `search_prog_...`. Splitting the stem on "_m" is unsafe:
    several names contain "_min_" or "_moment_", and "reference_mole_k6" contains "_m"
    inside "mole", which would collapse all four references onto a single key.
    """
    identifier = operator.metadata.get("program_id")
    if identifier: return f"{identifier}@{operator.cells}"
    stem = re.sub(r"_m\d+$", "", operator.name)
    tail = re.search(r"[0-9a-f]{8,}$", stem)
    return f"{tail.group(0) if tail else stem}@{operator.cells}"


operators, seen_keys = [], set()
for root, default_family in SOURCE_ROOTS:
    if not root.exists(): continue
    for candidate in load_operators(root):
        candidate.family = classify(candidate.name, default_family)
        key = program_key(candidate)
        # Release-first ordering: those copies carry the archive condition and the
        # exact-certificate metadata, so the first hit wins.
        if key in seen_keys: continue
        seen_keys.add(key)
        operators.append(candidate)

RESULTS_ROOT = PACKAGE_ROOT
inventory = pd.DataFrame([{
    "operator": op.name, "provenance": op.family, "cells": op.cells,
    "archive_condition": op.metadata.get("archive_condition"),
    "norm_class": op.metadata.get("norm_class", "diagonal"),
    "target_order": op.metadata.get("target_order"),
    "boundary_order": op.metadata.get("boundary_order") or op.metadata.get("left_boundary_order"),
    "model": op.metadata.get("model_id"),
    "has_norm": op.Q is not None,
} for op in operators]).sort_values(["provenance", "operator"]).reset_index(drop=True)

EXPECTED = {"promoted": 4, "reference": 4, "followup": 3}
counts = inventory.provenance.value_counts().to_dict()
print(f"{len(operators)} operators from {PACKAGE_ROOT.name}")
print("provenance:", counts)
for family, expected in EXPECTED.items():
    if counts.get(family, 0) != expected:
        print(f"  WARNING: expected {expected} {family} operators, found {counts.get(family, 0)}. "
              "If this cell was run after another loader, restart the kernel and run once.")
display(inventory)

18 operators from Verifier_Guided_Mimetic_Operators_Reproducibility_Package_Final_Clean
provenance: {'candidate': 7, 'promoted': 4, 'reference': 4, 'followup': 3}


,operator,provenance,cells,archive_condition,norm_class,target_order,boundary_order,model,has_norm
0,candidate_k2_b1_r1_s3_m200,candidate,200,None,diagonal,NaN,1,None,True
1,candidate_k4_b2_r4_s7_m200,candidate,200,None,diagonal,NaN,2,None,True
2,candidate_k6_b3_r8_s11_m200,candidate,200,None,diagonal,NaN,3,None,True
3,candidate_k8_b4_r9_s13_m200,candidate,200,None,diagonal,NaN,4,None,True
4,regression_prog_k6_block_psd_reflection_hybrid...,candidate,200,None,block_psd,6.0,3,None,True
5,regression_prog_k8_block_psd_asymmetric_hybrid...,candidate,200,None,block_psd,8.0,4,None,True
6,search_prog_k6_block_psd_reflection_hybrid_0cc...,candidate,200,None,block_psd,6.0,4,None,True
7,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,200,None,block_psd,6.0,3,None,True
8,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,200,None,block_psd,6.0,4,None,True
9,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,200,None,diagonal,6.0,3,None,True


## 2. Is a symmetric system available at all?

$\lVert Q_IL_I - L_I^\top Q_I\rVert_2 / \lVert Q_IL_I\rVert_2$ decides whether conjugate
gradients is defined for this operator. Positive definiteness of $-Q_IL_I$ is checked
separately, since both are required.

In [3]:
SYMMETRY_TOLERANCE = 1e-10


def weighted_symmetry_residual(L: np.ndarray, Q: np.ndarray) -> float:
    """Relative departure of Q L from symmetry: the quantity that decides everything."""
    product = Q @ L
    denominator = max(float(np.linalg.norm(product, 2)), 1e-300)
    return float(np.linalg.norm(product - product.T, 2) / denominator)


def symmetric_system(L: np.ndarray, Q: np.ndarray) -> dict[str, Any]:
    """Assemble -Q L and report whether it is a legitimate SPD system.

    Conjugate gradients is only defined for a symmetric positive-definite matrix. A
    certified weighted extended-Gauss identity delivers exactly that, since Q_I L_I =
    -S with S symmetric positive semidefinite. Without the identity, forcing the
    matrix to be symmetric silently replaces the problem with a different one, which
    the experiment below measures rather than assumes.
    """
    Q = 0.5 * (Q + Q.T)
    residual = weighted_symmetry_residual(L, Q)
    S = -(Q @ L)
    S_symmetric = 0.5 * (S + S.T)
    eigenvalues = np.linalg.eigvalsh(S_symmetric)
    return {
        "symmetry_residual": residual,
        "S": S,
        "S_symmetric_part": S_symmetric,
        "min_eigenvalue": float(eigenvalues[0]),
        "max_eigenvalue": float(eigenvalues[-1]),
        "condition_number": float(abs(eigenvalues[-1] / eigenvalues[0])) if eigenvalues[0] else float("inf"),
        "is_spd": bool(eigenvalues[0] > 0),
        "cg_applicable": bool(residual < SYMMETRY_TOLERANCE and eigenvalues[0] > 0),
    }

In [4]:
focus = [op for op in operators if op.family in ("promoted", "followup", "reference")]
availability = []
for op in focus:
    Q = op.interior_norm()
    if Q is None:
        availability.append({"operator": op.name, "family": op.family, "norm_available": False})
        continue
    system = symmetric_system(op.interior_laplacian(), Q)
    availability.append({"operator": op.name, "family": op.family, "norm_available": True,
                         **{k: system[k] for k in ("symmetry_residual", "min_eigenvalue",
                                                   "condition_number", "is_spd", "cg_applicable")}})
availability = pd.DataFrame(availability)
availability.to_csv(OUTPUT_DIR / "tables" / "symmetric_system_availability.csv", index=False)
display(availability)
print(f"CG applicable for {int(availability.cg_applicable.sum())} of {len(availability)} operators: "
      f"{availability.groupby('family').cg_applicable.sum().to_dict()}")

,operator,family,norm_available,symmetry_residual,min_eigenvalue,condition_number,is_spd,cg_applicable
0,full_metrics_prog_k6_block_psd_reflection_hybr...,promoted,True,1.061231e-16,0.049348,34299.193186,True,True
1,illumination_prog_k6_block_psd_reflection_hybr...,promoted,True,8.527437e-17,0.049348,24991.194988,True,True
2,no_archive_prog_k6_block_psd_asymmetric_hybrid...,promoted,True,1.017186e-16,0.049348,33767.720214,True,True
3,structure_only_prog_k6_block_psd_reflection_mi...,promoted,True,1.232072e-16,0.049348,37795.783702,True,True
4,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,True,1.099514e-16,0.049348,33159.040937,True,True
5,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,True,7.859843e-17,0.049348,50572.459192,True,True
6,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,True,9.223252e-17,0.049348,24991.162597,True,True
7,reference_mole_k2_m200,reference,True,7.195408e-02,0.049331,18763.712007,True,False
8,reference_mole_k4_m200,reference,True,1.485146e-01,0.048946,25516.438975,True,False
9,reference_mole_k6_m200,reference,True,5.223512e-01,-26.000485,53.040725,False,False


CG applicable for 7 of 11 operators: {'followup': 3, 'promoted': 4, 'reference': 0}


## 3. A manufactured Poisson problem

$-u'' = f$ on the unit interval with homogeneous Dirichlet data and
$u(x)=\sin(\pi x) + \tfrac14\sin(7\pi x)$, so that the solution carries both a smooth and a
moderately oscillatory component. Each operator solves the same problem by every route it
admits, to a relative residual of $10^{-10}$.

In [5]:
def manufactured_poisson(cells: int, modes=((1, 1.0), (7, 0.25))) -> tuple[np.ndarray, np.ndarray]:
    """Interior samples of u and of u'' for a Dirichlet-compatible manufactured field."""
    x = (np.arange(1, cells + 1) - 0.5) / cells
    u = sum(amplitude * np.sin(mode * math.pi * x) for mode, amplitude in modes)
    f = sum(-amplitude * (mode * math.pi) ** 2 * np.sin(mode * math.pi * x)
            for mode, amplitude in modes)
    return u, f


def _iterate(solver, A, b, tolerance, maxiter, M=None):
    """Run a SciPy iterative solver and record the true relative residual history."""
    history = []
    normalisation = max(float(np.linalg.norm(b)), 1e-300)

    def callback(xk):
        # SciPy's gmres callback reports a scalar residual by default while cg passes
        # the iterate; accept either so the histories are comparable.
        value = np.asarray(xk)
        if value.ndim == 0:
            history.append(float(value))
        else:
            history.append(float(np.linalg.norm(b - A @ value) / normalisation))

    keywords = {"maxiter": maxiter, "callback": callback}
    if solver is gmres and "callback_type" in inspect.signature(solver).parameters:
        keywords["callback_type"] = "pr_norm"
    if M is not None: keywords["M"] = M
    parameters = inspect.signature(solver).parameters
    keywords["rtol" if "rtol" in parameters else "tol"] = tolerance
    if solver is gmres and "restart" in parameters: keywords["restart"] = 50
    x, info = solver(A, b, **keywords)
    residual = float(np.linalg.norm(b - A @ x) / normalisation)
    return {"x": x, "info": int(info), "iterations": len(history),
            "final_relative_residual": residual, "history": history,
            "converged": bool(residual <= tolerance * 10)}


def poisson_experiment(operator, tolerance: float = 1e-10, maxiter: int = 5000) -> dict[str, Any]:
    """Solve one manufactured Poisson problem by every route each operator allows.

    Three routes are attempted for every operator so that the comparison is symmetric
    in effort rather than in outcome:

    * ``cg_symmetric`` -- conjugate gradients on -Q_I L_I. Legitimate only when that
      matrix is genuinely symmetric positive definite.
    * ``cg_forced`` -- conjugate gradients on the symmetric *part* of -Q_I L_I. Always
      runs; answers the original problem only when the residual above is negligible.
      Its solution error is the cost of pretending.
    * ``gmres`` and ``bicgstab`` -- non-symmetric iterations on L_I itself, the
      practical route when no symmetric system exists.
    """
    L = operator.interior_laplacian()
    Q = operator.interior_norm()
    cells = operator.cells
    u_exact, f = manufactured_poisson(cells)
    results: dict[str, Any] = {"operator": operator.name, "family": operator.family,
                               "cells": cells, "unknowns": int(L.shape[0])}
    if Q is None:
        results["norm_available"] = False
        return results
    system = symmetric_system(L, Q)
    results.update({k: system[k] for k in
                    ("symmetry_residual", "min_eigenvalue", "condition_number",
                     "is_spd", "cg_applicable")})
    results["norm_available"] = True

    def error_of(x):
        return float(np.linalg.norm(x - u_exact) / np.linalg.norm(u_exact))

    # Right-hand sides. L_I u = f, and the weighted form is (-Q L) u = -Q f.
    Q_symmetric = 0.5 * (Q + Q.T)
    b_weighted = -(Q_symmetric @ f)

    runs: dict[str, Any] = {}
    if system["cg_applicable"]:
        run = _iterate(cg, system["S"], b_weighted, tolerance, maxiter)
        runs["cg_symmetric"] = run
    forced = _iterate(cg, system["S_symmetric_part"], b_weighted, tolerance, maxiter)
    runs["cg_forced"] = forced
    for name, solver in (("gmres", gmres), ("bicgstab", bicgstab)):
        runs[name] = _iterate(solver, L, f, tolerance, maxiter)

    for name, run in runs.items():
        results[f"{name}_iterations"] = run["iterations"]
        results[f"{name}_converged"] = run["converged"]
        results[f"{name}_relative_residual"] = run["final_relative_residual"]
        results[f"{name}_solution_error"] = error_of(run["x"])
    results["_histories"] = {name: run["history"] for name, run in runs.items()}
    return results

In [6]:
poisson = [poisson_experiment(op) for op in focus]
poisson_table = pd.DataFrame([{k: v for k, v in row.items() if not k.startswith("_")}
                              for row in poisson])
poisson_table.to_csv(OUTPUT_DIR / "tables" / "poisson_solver_comparison.csv", index=False)
columns = [c for c in ["operator", "family", "symmetry_residual", "cg_applicable",
                       "cg_symmetric_iterations", "cg_symmetric_solution_error",
                       "cg_forced_iterations", "cg_forced_solution_error",
                       "gmres_iterations", "gmres_solution_error", "bicgstab_iterations"]
           if c in poisson_table]
display(poisson_table[columns])

,operator,family,symmetry_residual,cg_applicable,cg_symmetric_iterations,cg_symmetric_solution_error,cg_forced_iterations,cg_forced_solution_error,gmres_iterations,gmres_solution_error,bicgstab_iterations
0,full_metrics_prog_k6_block_psd_reflection_hybr...,promoted,1.061231e-16,True,116.0,1.658357e-07,116,1.658357e-07,269,1.664105e-07,144
1,illumination_prog_k6_block_psd_reflection_hybr...,promoted,8.527437e-17,True,115.0,5.871696e-06,115,5.871696e-06,1463,5.872741e-06,120
2,no_archive_prog_k6_block_psd_asymmetric_hybrid...,promoted,1.017186e-16,True,120.0,3.383585e-07,120,3.383585e-07,290,3.393034e-07,126
3,structure_only_prog_k6_block_psd_reflection_mi...,promoted,1.232072e-16,True,116.0,1.253234e-07,116,1.253234e-07,263,1.254110e-07,131
4,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,1.099514e-16,True,114.0,1.838518e-08,114,1.838521e-08,277,1.840534e-08,122
5,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,7.859843e-17,True,118.0,9.381541e-09,118,9.381509e-09,190,9.413121e-09,136
6,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,9.223252e-17,True,113.0,3.037647e-07,113,3.037647e-07,328,3.036869e-07,137
7,reference_mole_k2_m200,reference,7.195408e-02,False,NaN,NaN,102,1.050923e-03,1098,2.411721e-04,115
8,reference_mole_k4_m200,reference,1.485146e-01,False,NaN,NaN,108,2.406043e-02,237,3.471532e-07,146
9,reference_mole_k6_m200,reference,5.223512e-01,False,NaN,NaN,118,2.783656e-01,143,1.014780e-09,135


### The cost of pretending

`cg_forced` is the interesting column. It converges for every operator — a solver reports
success whether or not the matrix it was handed represents the intended problem. For the
certified operators the symmetric part *is* the system, so the answer is right. For the
references it is not, and the returned solution is wrong by an amount that has nothing to do
with discretization error.

In [7]:
comparison = poisson_table.copy()
comparison["forced_error_ratio"] = (comparison["cg_forced_solution_error"] /
                                    comparison["gmres_solution_error"])
summary_columns = ["operator", "family", "symmetry_residual", "cg_forced_solution_error",
                   "gmres_solution_error", "forced_error_ratio"]
display(comparison[summary_columns].sort_values("symmetry_residual"))

certified = comparison[comparison.family != "reference"]
references = comparison[comparison.family == "reference"]
print(f"Certified operators: forced-CG error {certified.cg_forced_solution_error.min():.2e} to "
      f"{certified.cg_forced_solution_error.max():.2e} (discretization level).")
print(f"References:          forced-CG error {references.cg_forced_solution_error.min():.2e} to "
      f"{references.cg_forced_solution_error.max():.2e}.")

,operator,family,symmetry_residual,cg_forced_solution_error,gmres_solution_error,forced_error_ratio
5,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,7.859843e-17,9.381509e-09,9.413121e-09,9.966417e-01
1,illumination_prog_k6_block_psd_reflection_hybr...,promoted,8.527437e-17,5.871696e-06,5.872741e-06,9.998221e-01
6,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,9.223252e-17,3.037647e-07,3.036869e-07,1.000256e+00
2,no_archive_prog_k6_block_psd_asymmetric_hybrid...,promoted,1.017186e-16,3.383585e-07,3.393034e-07,9.972153e-01
0,full_metrics_prog_k6_block_psd_reflection_hybr...,promoted,1.061231e-16,1.658357e-07,1.664105e-07,9.965457e-01
4,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,1.099514e-16,1.838521e-08,1.840534e-08,9.989064e-01
3,structure_only_prog_k6_block_psd_reflection_mi...,promoted,1.232072e-16,1.253234e-07,1.254110e-07,9.993013e-01
7,reference_mole_k2_m200,reference,7.195408e-02,1.050923e-03,2.411721e-04,4.357566e+00
8,reference_mole_k4_m200,reference,1.485146e-01,2.406043e-02,3.471532e-07,6.930781e+04
9,reference_mole_k6_m200,reference,5.223512e-01,2.783656e-01,1.014780e-09,2.743111e+08


Certified operators: forced-CG error 9.38e-09 to 5.87e-06 (discretization level).
References:          forced-CG error 1.05e-03 to 3.69e-01.


## 4. The same question recurs at every implicit time step

Backward Euler needs $(I-\Delta t\,L_I)$ solved once per step, so the availability of a
symmetric formulation is a recurring cost across a whole simulation rather than a one-off
property of a Poisson solve.

In [8]:
def implicit_heat_step(operator, dt_over_h2: float = 5.0, tolerance: float = 1e-10,
                       maxiter: int = 5000) -> dict[str, Any]:
    """One backward-Euler heat step, where the same symmetry question recurs.

    Implicit time stepping needs (I - dt L_I) solved at every step, so the presence or
    absence of a symmetric formulation is a per-step cost for the entire simulation
    rather than a one-off property of a Poisson solve.
    """
    L = operator.interior_laplacian()
    Q = operator.interior_norm()
    if Q is None: return {"operator": operator.name, "norm_available": False}
    Q = 0.5 * (Q + Q.T)
    h = 1.0 / operator.cells
    dt = dt_over_h2 * h * h
    A = np.eye(L.shape[0]) - dt * L
    weighted = Q @ A
    residual = float(np.linalg.norm(weighted - weighted.T, 2) /
                     max(float(np.linalg.norm(weighted, 2)), 1e-300))
    eigenvalues = np.linalg.eigvalsh(0.5 * (weighted + weighted.T))
    applicable = bool(residual < SYMMETRY_TOLERANCE and eigenvalues[0] > 0)
    u0, _ = manufactured_poisson(operator.cells)
    result = {"operator": operator.name, "family": operator.family, "norm_available": True,
              "dt_over_h2": dt_over_h2, "weighted_symmetry_residual": residual,
              "weighted_is_spd": bool(eigenvalues[0] > 0), "cg_applicable": applicable}
    if applicable:
        run = _iterate(cg, weighted, Q @ u0, tolerance, maxiter)
        result.update({"cg_iterations": run["iterations"], "cg_converged": run["converged"]})
    run = _iterate(gmres, A, u0, tolerance, maxiter)
    result.update({"gmres_iterations": run["iterations"], "gmres_converged": run["converged"]})
    return result

In [9]:
heat = pd.DataFrame([implicit_heat_step(op) for op in focus])
heat.to_csv(OUTPUT_DIR / "tables" / "implicit_heat_step.csv", index=False)
display(heat[[c for c in ["operator", "family", "weighted_symmetry_residual", "weighted_is_spd",
                          "cg_applicable", "cg_iterations", "gmres_iterations"] if c in heat]])

,operator,family,weighted_symmetry_residual,weighted_is_spd,cg_applicable,cg_iterations,gmres_iterations
0,full_metrics_prog_k6_block_psd_reflection_hybr...,promoted,8.475313e-17,True,True,51.0,19
1,illumination_prog_k6_block_psd_reflection_hybr...,promoted,8.908903e-17,True,True,54.0,26
2,no_archive_prog_k6_block_psd_asymmetric_hybrid...,promoted,1.389299e-16,True,True,49.0,21
3,structure_only_prog_k6_block_psd_reflection_mi...,promoted,8.878516e-17,True,True,51.0,19
4,v12_1_cold_prog_k6_block_psd_asymmetric_min_ne...,followup,1.066137e-16,True,True,41.0,18
5,v12_1_random_prog_k6_block_psd_reflection_min_...,followup,1.132017e-16,True,True,45.0,15
6,v12_diagonal_prog_k6_diagonal_asymmetric_hybri...,followup,6.485264e-17,True,True,47.0,25
7,reference_mole_k2_m200,reference,6.898020e-02,True,False,NaN,27
8,reference_mole_k4_m200,reference,1.436691e-01,True,False,NaN,17
9,reference_mole_k6_m200,reference,5.060009e-01,True,False,NaN,8


## 5. Figure

Left: true relative residual histories for one certified operator and the order-six reference.
Right: forced-CG solution error against the weighted-symmetry residual, which is the
relationship the experiment is really about.

In [10]:
certified_row = next(r for r in poisson if r["family"] != "reference" and r.get("cg_applicable"))
reference_row = next(r for r in poisson if "reference_mole_k6" in r["operator"])

figure, axes = plt.subplots(1, 2, figsize=(10.4, 4.1))
for label, history, style in (
        ("certified: CG on $-Q_IL_I$", certified_row["_histories"].get("cg_symmetric"), "-"),
        ("certified: GMRES on $L_I$", certified_row["_histories"].get("gmres"), "--"),
        ("reference k6: forced CG", reference_row["_histories"].get("cg_forced"), "-"),
        ("reference k6: GMRES on $L_I$", reference_row["_histories"].get("gmres"), "--")):
    if history: axes[0].semilogy(np.arange(1, len(history) + 1), history, style, linewidth=1.3,
                                 label=label)
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("relative residual")
axes[0].set_title("Residual histories"); axes[0].legend(fontsize=7)

for family, marker in (("reference", "s"), ("promoted", "o"), ("followup", "^")):
    subset = poisson_table[poisson_table.family == family]
    if subset.empty: continue
    axes[1].loglog(subset.symmetry_residual.clip(lower=1e-17),
                   subset.cg_forced_solution_error, marker, label=family, markersize=6)
axes[1].set_xlabel(r"weighted-symmetry residual"); axes[1].set_ylabel("forced-CG solution error")
axes[1].set_title("Pretending costs accuracy"); axes[1].legend(fontsize=7)
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "figures" / "solver_consequences.png", dpi=200)
plt.close(figure)
print("wrote", OUTPUT_DIR / "figures" / "solver_consequences.png")

wrote solver_experiment/figures/solver_consequences.png


## 6. Sentences for the manuscript

In [11]:
worst_reference = references.loc[references.cg_forced_solution_error.idxmax()]
k6 = comparison[comparison.operator.str.contains("reference_mole_k6")].iloc[0]
best_certified = certified.loc[certified.cg_symmetric_iterations.idxmin()]

report = {
    "package": str(PACKAGE_ROOT),
    "operators": int(len(focus)),
    "cg_applicable_certified": int(certified.cg_applicable.sum()),
    "cg_applicable_reference": int(references.cg_applicable.sum()),
    "certified_cg_iterations": {
        "min": int(certified.cg_symmetric_iterations.min()),
        "max": int(certified.cg_symmetric_iterations.max())},
    "reference_forced_cg_error": {row.operator: float(row.cg_forced_solution_error)
                                  for row in references.itertuples()},
    "certified_forced_cg_error_max": float(certified.cg_forced_solution_error.max()),
    "note": ("Iteration counts are not comparable across solver families; the claim concerns "
             "availability and correctness of the symmetric formulation."),
}
(OUTPUT_DIR / "solver_experiment_report.json").write_text(json.dumps(report, indent=2, default=str))

print("Sentences for the manuscript\n" + "-" * 66)
print(f"1. A symmetric positive-definite Poisson system is available for all "
      f"{report['cg_applicable_certified']} certified operators and for none of the "
      f"{len(references)} reference operators; conjugate gradients converges in "
      f"{report['certified_cg_iterations']['min']}-{report['certified_cg_iterations']['max']} "
      f"iterations on the certified systems.")
print(f"2. Symmetrising the reference anyway yields a convergent iteration that solves a "
      f"different problem: at order six the returned solution is in error by "
      f"{k6.cg_forced_solution_error:.2f} relative to the manufactured solution, against "
      f"{k6.gmres_solution_error:.1e} for a non-symmetric solve of the same operator, while "
      f"every certified operator stays at discretization level "
      f"(<= {report['certified_forced_cg_error_max']:.1e}).")
print(f"3. The same requirement recurs at every implicit time step, where the weighted "
      f"backward-Euler operator is symmetric positive definite for the certified constructions "
      f"and not for the references.")
print("-" * 66)

Sentences for the manuscript
------------------------------------------------------------------
1. A symmetric positive-definite Poisson system is available for all 7 certified operators and for none of the 4 reference operators; conjugate gradients converges in 113-120 iterations on the certified systems.
2. Symmetrising the reference anyway yields a convergent iteration that solves a different problem: at order six the returned solution is in error by 0.28 relative to the manufactured solution, against 1.0e-09 for a non-symmetric solve of the same operator, while every certified operator stays at discretization level (<= 5.9e-06).
3. The same requirement recurs at every implicit time step, where the weighted backward-Euler operator is symmetric positive definite for the certified constructions and not for the references.
------------------------------------------------------------------


## 7. Download the generated files

In [12]:
artifacts = sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file())
for path in artifacts: print(path)
bundle = Path(shutil.make_archive("solver_experiment_results", "zip",
                                  root_dir=".", base_dir=str(OUTPUT_DIR)))
print(f"\n{len(artifacts)} files, archive {bundle.name} ({bundle.stat().st_size / 1e3:.1f} kB)")
try:
    from google.colab import files
    files.download(str(bundle))
except Exception:
    try:
        from IPython.display import FileLink, display as _display
        _display(FileLink(str(bundle)))
    except Exception:
        print("Download the archive from:", bundle.resolve())

solver_experiment/figures/solver_consequences.png
solver_experiment/solver_experiment_report.json
solver_experiment/tables/implicit_heat_step.csv
solver_experiment/tables/poisson_solver_comparison.csv
solver_experiment/tables/symmetric_system_availability.csv

5 files, archive solver_experiment_results.zip (131.9 kB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>